# 02 · Second opinion: Best Practice Analyzer + Memory Analyzer

Runs Microsoft's **Best Practice Analyzer** (Semantic Link Labs, 60+ rules in five categories: Performance, DAX Expressions, Error Prevention, Maintenance, Formatting) and the **Memory Analyzer** (VertiPaq statistics) against the lab's semantic models.

Use it twice:
- **Before the fix step**: on `Customer 360 - As-Is` to record the baseline.
- **After the fix step**: on `Customer 360 - Working Copy` to compare.

This is the independent check on the agent: the agent explains and fixes; the analyzer counts. To use it on your own semantic model, change `WORKSPACE` and `MODELS` in the next cells to a dev or test copy.

In [ ]:
%pip install -q semantic-link-labs

In [ ]:
import sempy_labs as labs

WORKSPACE = "🕵️ Model Detective Agency"
MODELS = [
    "Customer 360 - As-Is",
    "Customer 360 - Working Copy",
]

In [ ]:
results = {}
for model in MODELS:
    try:
        df = labs.run_model_bpa(dataset=model, workspace=WORKSPACE, return_dataframe=True)
        results[model] = df
        print(f"{model}: {len(df)} rule violations")
        by = [c for c in ["Category", "Severity"] if c in df.columns]
        if by:
            display(df.groupby(by).size().reset_index(name="Violations"))
    except Exception as e:
        print(f"Skipped {model}: {e}")

### Memory Analyzer for the as-is semantic model
Look for the largest columns: the e-mail key, `row_hash` and the date-time columns should be near the top.

In [ ]:
labs.vertipaq_analyzer(dataset=MODELS[0], workspace=WORKSPACE)

### Memory Analyzer for the working copy (after the fix step)
Hiding columns and adding measures does not change the size. The point is to see the same columns, now hidden.

In [ ]:
labs.vertipaq_analyzer(dataset=MODELS[1], workspace=WORKSPACE)